# **Problem Statement**

## **Business Context**

ShopNest Global is a large-scale e-commerce platform operating across 30+ countries, serving over 50 million active customers and employing 2,000+ human support agents who run 24/7 across the US, Europe, India, and Southeast Asia, across product categories including electronics, fashion, groceries, and home appliances.

ShopNest processes over 200,000 orders per day. With every order comes the possibility of a delivery delay, a payment failure, a wrong item, or a return request. Customers reach out to the support team through tickets to report these issues and expect a fast, accurate resolution. These tickets are written in highly unstructured ways:

- Some are overloaded with background details where the real issue is buried.
- Others use abbreviations, shorthand, and order codes that are hard to interpret.
- Some contain so little information that the issue is entirely unclear.

As a result:

- Human agents spend the first **2–3 minutes** on each ticket just decoding what the customer is asking before any resolution work can begin.
- During peak periods (sales, holidays, logistics disruptions), daily ticket volume can spike from ~5,000 to **~15,000**, multiplying this inefficiency.
- High volume and inconsistent ticket content contribute to agent fatigue and higher error rates when accuracy is most critical.
- Drafting responses is manual, slow, and inconsistent in tone and clarity across agents, leading to suboptimal customer experiences.

## **Objective**

The objective is to build a POC of an AI-powered ticket intelligence system for ShopNest Global that:

1. **Summarises** incoming raw, unstructured tickets into a clean, concise summary for the support agent.
2. **Evaluates** the generated summary using an LLM-as-Judge approach, scoring quality on defined criteria.
3. **Generates** a professional, empathetic customer response grounded in ShopNest's support policies.
4. **Evaluates** the generated response using an LLM-as-Judge approach, scoring resolution quality.
5. **Compiles** all outputs into a single structured table and exports it for downstream use.

The end goal is to demonstrate that AI-assisted summarisation and response generation can meaningfully improve the consistency and quality of customer support operations at scale.

## **Data Dictionary**

| Column Name         | Data Type | Description                                                       |
| ------------------- | --------- | ----------------------------------------------------------------- |
| support_ticket_id   | Integer   | Unique identifier assigned to each support ticket                 |
| support_ticket_text | String    | Free-form text describing the issue or request raised by the user |

# **Please read the instructions carefully before starting the project.**

This is a commented Python Notebook file in which all the instructions and tasks to be performed are mentioned.

* Blanks '\_\_\_\_\_' are provided in the notebook that
needs to be filled with an appropriate code to get the correct result. With every '\_\_\_\_\_' blank, there is a comment that briefly describes what needs to be filled in the blank space.
* Identify the task to be performed correctly, and only then proceed to write the required code.
* Please sequentially run the code cells from the beginning to avoid any unnecessary errors.
* Add the results/observations derived from the analysis in the presentation and submit the same. Any mathematical or computational details that are a graded part of the project can be included in the Appendix section of the presentation.

# **Installing and Importing Necessary Libraries**

In [2]:
# Install LangChain and OpenAI for LLM API access, and pandas for data handling
# Pinned versions ensure reproducibility across environments
%pip install pandas==2.2.2 langchain-openai==1.1.12 openai==2.31.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 40.0 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached numpy-2.5.1-cp314-cp314-macosx_14_0_arm64.whl.metadata (6.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 43.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 31.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.6/731.6 kB 32.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 64.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.6/983.6 kB 54.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 562.2/562.2 kB 28.2 MB/s  0:00:00
Using cached numpy-2.5.1-cp314-cp314-macosx_14_0_arm64.whl (5.3 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 640.7/640.7 kB 31.9 MB/s  0:00:00
  Created wheel for pandas: filename=pandas-2.2.2-cp314-cp314

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for VSCode), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [1]:
# Core libraries
import os            # For environment variable access (API key fallback)
import json
import re            # For parsing structured evaluator output into numeric scores
import pandas as pd  # For loading, manipulating, and exporting tabular data
from langchain_openai import ChatOpenAI  # OpenAI
from openai import OpenAI

## **Loading the Open API Key**

In [2]:
# Load the JSON file and extract values
file_name = 'config.json'                                                       # Name of the configuration file
with open(file_name, 'r') as file:                                              # Open the config file in read mode
    config = json.load(file)                                                    # Load the JSON content as a dictionary
    OPENAI_API_KEY = config.get("OPENAI_API_KEY")                                             # Extract the API key from the config
    OPENAI_API_BASE = config.get("OPENAI_API_BASE")
# Store API credentials in environment variables
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY                                  # Set API key as environment variable
os.environ["OPENAI_BASE_URL"] = OPENAI_API_BASE                                 # Set API base URL as environment variable
client = OpenAI()

# **Data Loading**

## **Load the data**

In [ ]:
# Uncomment the lines below if the dataset is stored in Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

In [5]:
# Load the support ticket dataset from a CSV file
# A deep copy is made so the original data remains unchanged throughout the notebook
df = pd.read_csv("/Users/prasadimmadi/Desktop/Git Local/Project_AgenticAI/Project_1_Support_Ticket_Analysis/support_ticket_data.csv")         # complete the code to add the path to the CSV data file

print(f"Dataset loaded successfully. Shape: {df.shape}")

Dataset loaded successfully. Shape: (30, 2)


## **Data Overview**

In [6]:
df.head()                   # View the first 5 rows of the data

,support_ticket_id,support_ticket_desc
0,1,I cannot believe the level of service I have r...
1,2,"Ord SNX-8902 ACH debit failed at checkout, tri..."
2,3,not working. please help.
3,4,Okay so I have genuinely had it with this comp...
4,5,refund not received


In [7]:
df.tail()                   # View the last 5 rows of the data

,support_ticket_id,support_ticket_desc
25,26,"Hi, just wanted to check on the status of my o..."
26,27,"Hello, quick question - what is your standard ..."
27,28,Hi there. I received my order SNX-7823 today a...
28,29,"Hey, I placed an order yesterday evening - ord..."
29,30,"Hi, I was just checking whether my recent retu..."


# **Summarization**

## **Set up an LLM**

In [8]:
# uncomment one of the following code snippets to choose the LLM for output generation
MODEL_NAME = "gpt-4o-mini"
# MODEL_NAME = "gpt-4o"

## **Set parameters**

In [9]:
SUMMARY_TEMP = 0.2   # Complete the code to choose the temperature value for the summarization task

## **Prompting Technique**

### **System Message**

In [ ]:
SUMMARISER_SYSTEM = """


# WRITE YOUR SYSTEM MESSAGE HERE


"""

In [ ]:
SUMMARISER_USER = """


# WRITE YOUR USER MESSAGE HERE


Ticket: "{ticket}"
Summary:
"""

**Note**: DO NOT remove the following part in the `SUMMARISER_USER` variable:

```
Ticket: "{ticket}"
Summary:
```

### **Generate Ticket Summaries**

In [ ]:
def generate_summary(ticket: str) -> str:
    # Inject the actual ticket into the few-shot template
    user_message = SUMMARISER_USER.format(ticket=ticket)

    # Call the Groq API with the summariser system message and formatted user message
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=SUMMARY_TEMP,       # Low temperature for consistent, accurate output
        messages=[
            {"role": "system", "content": SUMMARISER_SYSTEM},
            {"role": "user",   "content": user_message}
        ]
    )
    # Extract and return the model's text output, stripping leading/trailing whitespace
    return response.choices[0].message.content.strip()

In [ ]:
test_ticket = df["support_ticket_desc"].iloc[_____]    # Complete the code to add the index of any one ticket to test the summariser before running the full pipeline

print("INPUT TICKET:")
print("-" * 60)
print(test_ticket)

print("\nGENERATED SUMMARY:")
print("-" * 60)
test_summary = generate_summary(test_ticket)
print(test_summary)

# **Evaluation for Summarization**

## **Setup an LLM**

In [ ]:
# uncomment one of the following code snippets to choose the LLM for output generation
# MODEL_NAME = "gpt-4o-mini"
# MODEL_NAME = "gpt-4o"

## **System Message**

In [ ]:
# ── Summarisation Evaluator System Message ──────────────────────────────────
SUMMARY_EVAL_SYSTEM = """


# WRITE YOUR EVALUATION SYSTEM MESSAGE HERE


"""

In [ ]:
# ── Evaluator User Message Template ─────────────────────────────────────────
SUMMARY_EVAL_USER = """\
Original Ticket: "{ticket}"
Ticket Summary: "{summary}"
Evaluate the summary."""

**Note**: DO NOT remove the following part in the `SUMMARY_EVAL_USER` variable:

```
Original Ticket: "{ticket}"
Ticket Summary: "{summary}"
Evaluate the summary.
```

## **Generate Evaluation Scores**

In [ ]:
def evaluate_summary(ticket: str, summary: str) -> str:
    # Inject the ticket and summary into the evaluation user message template
    user_message = SUMMARY_EVAL_USER.format(ticket=ticket, summary=summary)

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": SUMMARY_EVAL_SYSTEM},
            {"role": "user",   "content": user_message}
        ]
    )
    return response.choices[0].message.content.strip()

In [ ]:
# Run the summarisation evaluator on the test ticket and its generated summary
# This validates the evaluator output format before the full pipeline run

print("SUMMARISATION EVALUATION")
print("=" * 60)

test_sum_eval_raw = evaluate_summary(test_ticket, test_summary)
print(test_sum_eval_raw)

# **Response Generation**

## **Setup an LLM**

In [ ]:
# uncomment one of the following code snippets to choose the LLM for output generation
# MODEL_NAME = "gpt-4o-mini"
# MODEL_NAME = "gpt-4o"

## **Set parameters**

In [ ]:
GENERATION_TEMP = _____   # Complete the code to choose the temperature value for the response generation task

## **Prompting Technique**

### **System Message**

In [ ]:
# ── Response Generator System Message ────────────────────────────────────────
# The output format is strictly defined at the bottom so responses are
# consistent and ready to send to customers with minimal human agent editing.

GENERATOR_SYSTEM = """\


# WRITE YOUR SYSTEM MESSAGE HERE


"""

In [ ]:
# ── User Message Template ──────────────────────────────────────────────

GENERATOR_USER = """\

# WRITE THE USER MESSAGE HERE

Ticket Summary: "{summary}"
"""

**Note**: DO NOT remove the following part in the `GENERATOR_USER` variable:

```
Ticket Summary: "{summary}
```

### **Generate a User Response**

In [ ]:
def generate_response(summary: str) -> str:
    # Inject the summary into the CoT user message template
    user_message = GENERATOR_USER.format(summary=summary)

    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=GENERATION_TEMP,       # Moderate temperature for natural, empathetic tone
        messages=[
            {"role": "system", "content": GENERATOR_SYSTEM},
            {"role": "user",   "content": user_message}
        ]
    )
    return response.choices[0].message.content.strip()

In [ ]:
# Test the response generator on the sample ticket summary
# Validate that the output follows the format and stays within policy boundaries

print("INPUT SUMMARY:")
print("-" * 60)
print(test_summary)

print("\nGENERATED RESPONSE:")
print("-" * 60)
test_response = generate_response(test_summary)
print(test_response)

# **Evaluation for Response Generation**

## **Setup an LLM**

In [ ]:
# uncomment one of the following code snippets to choose the LLM for output generation
# MODEL_NAME = "gpt-4o-mini"
# MODEL_NAME = "gpt-4o"

## **System Message**

In [ ]:
# ── Response Evaluator System Message
RESPONSE_EVAL_SYSTEM = """


# WRITE YOUR SYSTEM MESSAGE HERE


"""

In [ ]:
# ── Response Evaluator User Message Template
RESPONSE_EVAL_USER = """\
Ticket Summary: "{summary}"
Generated Response: "{response}"
Evaluate the response."""

**Note**: DO NOT remove the following part in the `RESPONSE_EVAL_USER` variable:

```
Ticket Summary: "{summary}"
Generated Response: "{response}"
Evaluate the response.
```

## **Generate Evaluation Scores**

In [ ]:
def evaluate_response(summary: str, response_text: str) -> str:
    # Inject the summary and generated response into the evaluator user message
    user_message = RESPONSE_EVAL_USER.format(summary=summary, response=response_text)

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": RESPONSE_EVAL_SYSTEM},
            {"role": "user",   "content": user_message}
        ]
    )
    return response.choices[0].message.content.strip()

In [ ]:
# Run the response evaluator on the test summary and generated response
# Validates the format and logic of the evaluator before the full pipeline run

print("RESPONSE EVALUATION")
print("=" * 60)

test_resp_eval_raw = evaluate_response(test_summary, test_response)
print(test_resp_eval_raw)

# **Output & Compilation**

In [ ]:
# Initialise storage lists - one entry per ticket, in the same order as the DataFrame
summaries        = []   # Generated summaries
sum_eval  = []   # Parsed summarisation evaluation dicts
responses        = []   # Generated customer responses
resp_eval = []   # Parsed response evaluation dicts

In [ ]:
for idx, row in df.iterrows():
    ticket_id   = row["support_ticket_id"]
    description = row["support_ticket_desc"]

    # ── Step 1: Generate summary ──────────────────────────────────────────────
    # Few-shot prompting extracts the core issue, order ref, product, and resolution request
    summary = generate_summary(description)
    summaries.append(summary)

    # ── Step 2: Evaluate the summary ─────────────────────────────────────────
    # LLM-as-Judge scores Information Extraction and Field Coverage (each 1-3)
    sum_eval_raw = sum_eval.append(evaluate_summary(description, summary))

    # ── Step 3: Generate customer response ───────────────────────────────────
    # CoT prompting with embedded policies produces a policy-aligned customer reply
    response = generate_response(summary)
    responses.append(response)

    # ── Step 4: Evaluate the response ────────────────────────────────────────
    # LLM-as-Judge scores Issue Addressal and Resolution Clarity (each 1-3)
    resp_eval_raw = resp_eval.append(evaluate_response(summary, response))

    print(f"  Ticket {ticket_id} - done")

print(f"\nPipeline complete. Processed {len(df)} tickets.")

## **Combine All Outputs into a Single Consolidated Table**

In [ ]:
# ── Build the core output columns ────────────────────────────────────────────
# Required by the rubric: ticket ID, raw ticket text, generated summary, generated response
output_df = pd.DataFrame({
    "support_ticket_id"  : df["support_ticket_id"].values,
    "support_ticket_desc": df["support_ticket_desc"].values,
    "generated_summary"  : summaries,
    "generated_response" : responses
})

In [ ]:
# View the key columns for the first 5 tickets in the output table
# This confirms that all pipeline outputs are aligned correctly row by row
output_df.head(5)

In [ ]:
# Export the full output table to a CSV file
# index=False ensures the DataFrame index is not written as a column

output_path = "_____"                                  # Complete the code to provide the name for the CSV file
output_df.to_csv(output_path, index=False)                       # Save the file

print(f"Output saved to       : {output_path}")

### Save the Evaluation Scores for Summarization and Response Generation

In [ ]:
# --- Process Summarization Evaluations (sum_eval) ---
summary_evaluations = []
for eval_json_str in sum_eval:
    eval_data = json.loads(eval_json_str)
    scores = eval_data.get('scores', {})
    overall_verdict = eval_data.get('overall_verdict')

    evaluation_entry = {
        'technical_accuracy': scores.get('technical_accuracy'),
        'completeness': scores.get('completeness'),
        'conciseness': scores.get('conciseness'),
        'hallucination_check': scores.get('hallucination_check'),
        'summary_overall_verdict': overall_verdict
    }
    summary_evaluations.append(evaluation_entry)

summary_eval_df = pd.DataFrame(summary_evaluations)

# --- Process Response Evaluations (resp_eval) ---
response_evaluations = []
for eval_json_str in resp_eval:
    eval_data = json.loads(eval_json_str)
    scores = eval_data.get('scores', {})
    feedback = eval_data.get('feedback', {})
    overall_verdict = eval_data.get('overall_verdict')

    evaluation_entry = {
        'alignment_with_summary': scores.get('alignment_with_summary'),
        'actionability': scores.get('actionability'),
        'tone_empathy': scores.get('tone_empathy'),
        'policy_compliance': scores.get('policy_compliance'),
        'response_overall_verdict': overall_verdict,
        'feedback_strengths': str(feedback.get('strengths')),
        'feedback_weaknesses': str(feedback.get('weaknesses')),
        'feedback_risk_factors': str(feedback.get('risk_factors'))
    }
    response_evaluations.append(evaluation_entry)

response_eval_df = pd.DataFrame(response_evaluations)

In [ ]:
display(summary_eval_df.head())
summary_eval_df.to_csv("_____", index=False)     # Complete the code to provide the name for the CSV file

In [ ]:
display(response_eval_df.head())
response_eval_df.to_csv("_____", index=False)     # Complete the code to provide the name for the CSV file

# **Business Insights & Recommendations**

## **Business Insights**

- Add to the presentation

## **Recommendations**

- Add to the presentation

<font size=6>Power Ahead!</font>
___